In [ ]:
!pip install -q datasets pandas

In [ ]:
from datasets import load_dataset
import pandas as pd

HF_REPO_ID = "dinushiTJ/nz_research_commons_classification"

dataset = load_dataset(HF_REPO_ID)

print(dataset)

README.md:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/83.2M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/13271 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3318 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['title', 'authors', 'subjects', 'abstract', 'text', 'record_id_hash', 'prompt', 'token_count', 'classification_label', 'classification_reason', 'year', 'classification_confidence', 'embedding_text', 'embedding_token_count', 'conversations'],
        num_rows: 13271
    })
    test: Dataset({
        features: ['title', 'authors', 'subjects', 'abstract', 'text', 'record_id_hash', 'prompt', 'token_count', 'classification_label', 'classification_reason', 'year', 'classification_confidence', 'embedding_text', 'embedding_token_count', 'conversations'],
        num_rows: 3318
    })
})


In [ ]:
# Use the first available split
split_name = list(dataset.keys())[0]
df = dataset[split_name].to_pandas()

print("Using split:", split_name)
print("Total records:", len(df))
print("Columns:", df.columns.tolist())
print(df.head())

Using split: train
Total records: 13271
Columns: ['title', 'authors', 'subjects', 'abstract', 'text', 'record_id_hash', 'prompt', 'token_count', 'classification_label', 'classification_reason', 'year', 'classification_confidence', 'embedding_text', 'embedding_token_count', 'conversations']
                                               title  \
0  The Te Aroha hot springs (mainly in the ninete...   
1                       Jespersen cycles on Malekula   
2  Fabric, fitness, and femininity:  A new materi...   
3  On the supply-side of determinants of regional...   
4  The Role of Fatherhood in the Everyday Life of...   

                           authors         subjects  \
0                     Hart, Philip             None   
1             Barbour, Julie Renee        diachrony   
2           Brice, Julie Elizabeth       Activewear   
3                   McCann, Philip  economic theory   
4  Elkington, Angus Marangai Brons        maori men   

                                         

In [ ]:
# Use first available split
split_name = list(dataset.keys())[0]
ds = dataset[split_name]

print("Using split:", split_name)
print("Total records:", len(ds))
print(ds.features)

Using split: train
Total records: 13271
{'title': Value('string'), 'authors': Value('string'), 'subjects': Value('string'), 'abstract': Value('string'), 'text': Value('string'), 'record_id_hash': Value('string'), 'prompt': Value('string'), 'token_count': Value('int32'), 'classification_label': ClassLabel(names=['maori_origin', 'non_maori_origin']), 'classification_reason': Value('string'), 'year': Value('string'), 'classification_confidence': Value('float32'), 'embedding_text': Value('string'), 'embedding_token_count': Value('int64'), 'conversations': List({'role': Value('string'), 'content': Value('string')})}


In [ ]:
LABEL_COL = "classification_label"

# Automatically convert label ids to label names using Hugging Face metadata
label_feature = ds.features[LABEL_COL] # FIX: Changed `dataset` to `ds`

if hasattr(label_feature, "int2str"):
    df["label_text"] = [
        label_feature.int2str(int(x)) if pd.notna(x) else None
        for x in df[LABEL_COL]
    ]
else:
    # If labels are already strings, just copy them
    df["label_text"] = df[LABEL_COL].astype(str)

print("Original label values:")
print(df[LABEL_COL].value_counts(dropna=False))

print("\nAutomatically converted label values:")
print(df["label_text"].value_counts(dropna=False))

Original label values:
classification_label
1    11802
0     1469
Name: count, dtype: int64

Automatically converted label values:
label_text
non_maori_origin    11802
maori_origin         1469
Name: count, dtype: int64


# Reproducible random method to Select Sample size maori/non-maori 600/600


In [ ]:
RANDOM_SEED = 42
N_MAORI = 600
N_NON_MAORI = 600

maori_indices = df[df[SAMPLE_LABEL_COL] == "maori_origin"].index # FIX: Use df[SAMPLE_LABEL_COL]
non_maori_indices = df[df[SAMPLE_LABEL_COL] == "non_maori_origin"].index # FIX: Use df[SAMPLE_LABEL_COL]

print("Available records before sampling:")
print("maori_origin:", len(maori_indices))
print("non_maori_origin:", len(non_maori_indices))

if len(maori_indices) < N_MAORI:
    raise ValueError(
        f"Not enough maori_origin records. Available: {len(maori_indices)}, requested: {N_MAORI}"
    )

if len(non_maori_indices) < N_NON_MAORI:
    raise ValueError(
        f"Not enough non_maori_origin records. Available: {len(non_maori_indices)}, requested: {N_NON_MAORI}"
    )

Available records before sampling:
maori_origin: 1469
non_maori_origin: 11802


In [ ]:
# Balanced stratified random sampling without replacement.
# This randomly selects 600 indices from each label stratum.
maori_sample_indices = pd.Series(maori_indices).sample(
    n=N_MAORI,
    random_state=RANDOM_SEED,
    replace=False
).tolist()

non_maori_sample_indices = pd.Series(non_maori_indices).sample(
    n=N_NON_MAORI,
    random_state=RANDOM_SEED,
    replace=False
).tolist()

selected_indices = maori_sample_indices + non_maori_sample_indices

cross_provider_sample_1200 = df.loc[selected_indices].copy()

# Shuffle final sample rows, but do not add any new columns.
cross_provider_sample_1200 = cross_provider_sample_1200.sample(
    frac=1,
    random_state=RANDOM_SEED,
    replace=False
).reset_index(drop=True)

print("Final sample shape:", cross_provider_sample_1200.shape)
print("Final columns:")
print(cross_provider_sample_1200.columns.tolist())

Final sample shape: (1200, 16)
Final columns:
['title', 'authors', 'subjects', 'abstract', 'text', 'record_id_hash', 'prompt', 'token_count', 'classification_label', 'classification_reason', 'year', 'classification_confidence', 'embedding_text', 'embedding_token_count', 'conversations', 'label_text']


In [ ]:
# Show final sample counts without adding any new column
if hasattr(label_feature, "int2str"):
    final_label_text = pd.Series(
        [
            label_feature.int2str(int(x)) if pd.notna(x) else None
            for x in cross_provider_sample_1200[LABEL_COL]
        ]
    )
else:
    final_label_text = cross_provider_sample_1200[LABEL_COL].astype(str)

print("Final sample label counts:")
print(final_label_text.value_counts(dropna=False))

print("\nOriginal classification_label counts in sample:")
print(cross_provider_sample_1200[LABEL_COL].value_counts(dropna=False))

Final sample label counts:
non_maori_origin    600
maori_origin        600
Name: count, dtype: int64

Original classification_label counts in sample:
classification_label
1    600
0    600
Name: count, dtype: int64


In [ ]:
if "record_id_hash" in cross_provider_sample_1200.columns:
    duplicate_count = cross_provider_sample_1200["record_id_hash"].duplicated().sum()
    print("Duplicate record_id_hash count:", duplicate_count)
else:
    print("record_id_hash column not found, duplicate check skipped.")

Duplicate record_id_hash count: 0


In [ ]:
sample_dataset = Dataset.from_pandas(
    cross_provider_sample_1200,
    preserve_index=False
)

print(sample_dataset)
print(sample_dataset.features)

Dataset({
    features: ['title', 'authors', 'subjects', 'abstract', 'text', 'record_id_hash', 'prompt', 'token_count', 'classification_label', 'classification_reason', 'year', 'classification_confidence', 'embedding_text', 'embedding_token_count', 'conversations', 'label_text'],
    num_rows: 1200
})
{'title': Value('string'), 'authors': Value('string'), 'subjects': Value('string'), 'abstract': Value('string'), 'text': Value('string'), 'record_id_hash': Value('string'), 'prompt': Value('string'), 'token_count': Value('int32'), 'classification_label': Value('int64'), 'classification_reason': Value('string'), 'year': Value('string'), 'classification_confidence': Value('float32'), 'embedding_text': Value('string'), 'embedding_token_count': Value('int64'), 'conversations': List({'content': Value('string'), 'role': Value('string')}), 'label_text': Value('string')}


In [ ]:
OUTPUT_REPO_ID = "dinushiTJ/nz_research_commons_cross_provider_validation_sample_1200"

sample_dataset.push_to_hub(
    OUTPUT_REPO_ID,
    private=False
)

print("Pushed sample dataset to:")
print(f"https://huggingface.co/datasets/{OUTPUT_REPO_ID}")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  99%|#########8| 7.36MB / 7.46MB            

README.md:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

Pushed sample dataset to:
https://huggingface.co/datasets/dinushiTJ/nz_research_commons_cross_provider_validation_sample_1200
